# Real MNIST: Load, Clean, and Visualize

**Requires internet access to run.** This notebook downloads the real MNIST data set (~70,000 images) from OpenML the first time you run the load cell -- scikit-learn caches it locally afterward, so later runs are fast. If you're in an offline/sandboxed environment, the fetch cell below will fail; run this in a normal Jupyter install, Google Colab, or anywhere with internet instead.

**What is MNIST?** A famous benchmark set of 70,000 grayscale images of handwritten digits (0-9), each 28x28 pixels, each labeled with the digit it actually shows. It's one of the most widely used "hello world" data sets for image classification.

**How are the images stored?** Each image is a 28x28 grid of pixel-brightness numbers, each ranging from 0 (black) to 255 (white) -- the standard scale for 8-bit grayscale images.

*Note: this notebook's outputs are empty/unexecuted, since it can't actually be run in the sandbox that generated it. Run all cells yourself to populate real results.*


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_openml

# fetch_openml downloads (or loads from local cache, after the first run)
# the "mnist_784" data set -- named for its 784 pixels per image, since
# 28 x 28 = 784. Setting as_frame=True hands it back to us already as a
# pandas DataFrame: one row per image, one column per pixel (784 columns
# total), which is exactly the table shape we want.
print("Fetching MNIST from OpenML (this can take a minute on first run)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=True)

digits_df = mnist.data.copy()                    # the 784 pixel columns
digits_df["label"] = mnist.target.astype(int)    # the true digit, 0-9

pixel_columns = [c for c in digits_df.columns if c != "label"]

print(f"Loaded {len(digits_df)} handwritten digit images.")
print(f"Each image has {len(pixel_columns)} pixels (28x28 grid, flattened).")
digits_df[["label"] + pixel_columns[:5]].head()


## Cleaning step: drop everything except digits 3 and 8

In plain terms: we're throwing out every image except the ones labeled "3" or "8", leaving a smaller data set that only contains those two digits. This is a common real-world step when you want to simplify a problem -- 3 and 8 are also a classic hard pair to tell apart by eye (and for models), since they can look visually similar when handwritten.


In [ ]:
before_count = len(digits_df)
digits_df = digits_df[digits_df["label"].isin([3, 8])].reset_index(drop=True)
after_count = len(digits_df)

print(f"Dropped {before_count - after_count} images that weren't a 3 or an 8.")
print(f"{after_count} images remain.")
print(digits_df["label"].value_counts())


## Visualize: a grid of sample digit images

Each row in our table is really a 28x28 image squashed into one long row of 784 numbers. To *see* it as a picture again, we reshape each row back into a 28x28 grid and hand it to matplotlib's `imshow`, which turns a grid of numbers into a grid of grayscale pixels (low numbers = dark, high numbers = light). We do this for a handful of sample images side by side so we can visually inspect how differently people write the same digit -- some 3s and 8s are neat, some are messy or oddly shaped.


In [ ]:
n_rows, n_cols = 4, 8  # show 32 example images total, mixed 3s and 8s
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.2, n_rows * 1.4))

# Grab a mix of examples: half from digit 3, half from digit 8.
samples_per_digit = (n_rows * n_cols) // 2
threes = digits_df[digits_df["label"] == 3].head(samples_per_digit)
eights = digits_df[digits_df["label"] == 8].head(samples_per_digit)
sample_df = pd.concat([threes, eights]).reset_index(drop=True)

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_grid = row[pixel_columns].values.astype(float).reshape(28, 28)  # un-flatten
    ax.imshow(image_grid, cmap="gray_r")  # gray_r: high pixel value = dark ink
    ax.set_title(str(row["label"]), fontsize=10)
    ax.axis("off")  # hide axis ticks/numbers, we just want the picture

fig.suptitle("Sample handwritten digits (real MNIST): 3s and 8s", fontsize=13)
fig.tight_layout()
plt.show()
